In [1]:
from src.citibike.citibike_utils import get_trip_duration_mins
from src.utils.datetime_utils import timestamp_to_date_col

from pyspark.sql.functions import create_map, lit, col

In [2]:
df = spark.read.table("citibike_dev.01_bronze.jc_citibike")

In [3]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")

In [4]:
df = timestamp_to_date_col(spark, df, "started_at", "trip_start_date")

In [5]:
df = df.withColumn("metadata",
                   create_map(
                       lit("pipeline_id"), lit("placeholder"),
                       lit("run_id"), lit("placeholder"),
                       lit("task_id"), lit("placeholder"),
                       lit("processed_timestamp"), lit("placeholder"),
                   ))

In [6]:
df = df.select("ride_id",
               "trip_start_date", 
               "started_at", 
               "ended_at", 
               "start_station_name",
               "end_station_name",
               "trip_duration_mins", 
               "metadata")

In [7]:
df.write\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("citibike_dev.02_silver.jc_citibike")